In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
sys.path.append("..")
from src.pipeline import RUTA_LIMPIA
df = pd.read_parquet(RUTA_LIMPIA)

In [ ]:
# función para encontrar nombre de empresa
# cambiar key por nombre de la empresa sin la primera letra mayúscula
key="ura" 
[n for n in df["NOMBRE_CORTO"].unique() if key in n.lower()]

### Encontramos Sura como "Seguros Sura"

In [ ]:
# Ejemplo
aseguradora="Seguros Sura"
anio=2025
estado="Ciudad de México"
ramo="Vida"

In [ ]:
from metricas import filtrar
filtrar(df,aseguradora,ramo,estado,anio=anio)


### Perfil de la aseguradora: su participación en cada ramo

`participacion_mercado` no lleva filtros propios: primero acotas con `filtrar`,
luego calculas. La regla es **no filtrar por la aseguradora**, porque el total
del mercado es el denominador. Si la filtras antes, te da 100%.

Tampoco filtramos por ramo: queremos todos, para ver el perfil completo.

In [ ]:
from metricas import participacion_mercado

# El mercado: todos los ramos del año, SIN filtrar por aseguradora.
mercado = filtrar(df, anio=anio)

# por=("DESC_RAMO",) es el default: una fila por ramo × institución.
part = participacion_mercado(mercado)
part.shape

Ahora sí, nos quedamos con nuestra aseguradora. Salen solo los ramos donde
opera; los que no aparecen es porque no emitió prima ese año.

In [ ]:
perfil = (filtrar(part, institucion=aseguradora)
          .sort_values("participacion_pct", ascending=False, ignore_index=True))

perfil[["DESC_RAMO", "prima_mdp", "mercado_mdp", "participacion_pct"]].round(
    {"prima_mdp": 1, "mercado_mdp": 1, "participacion_pct": 2}
)

Ordenar por `participacion_pct` responde *dónde pesa más* la aseguradora.
Ordenar por `prima_mdp` responde *dónde tiene más dinero*.

In [ ]:
perfil.sort_values("prima_mdp", ascending=False, ignore_index=True)[
    ["DESC_RAMO", "prima_mdp", "participacion_pct"]
].round({"prima_mdp": 1, "participacion_pct": 2})

---
## Siniestralidad contra el mercado

`comparar_vs_mercado` corre la misma métrica dos veces —sobre la aseguradora y
sobre el mercado completo— y las pega lado a lado. Nota que `por` NO lleva la
institución: es justamente la dimensión contra la que comparamos.

La diferencia entre dos columnas `_pct` está en **puntos porcentuales**.

In [ ]:
from metricas import siniestralidad, frecuencia_severidad, crecimiento_anual, comparar_vs_mercado

sin = comparar_vs_mercado(mercado, aseguradora, siniestralidad)

sin[["DESC_RAMO", "prima_mdp_inst",
     "siniestralidad_pct_inst", "siniestralidad_pct_mercado", "siniestralidad_pct_dif"]] \
   .sort_values("siniestralidad_pct_dif", ascending=False, ignore_index=True) \
   .round(2)

Un `_dif` positivo significa que Sura paga más siniestros por peso emitido que
el promedio del ramo. Es el signo "malo": peor resultado técnico.

### Por qué: frecuencia o severidad

La siniestralidad alta puede venir de **muchos** siniestros o de siniestros
**caros**. `frecuencia_severidad` separa las dos causas.

Ojo con el denominador de la frecuencia: `RIESGOS_ASEG_VIG` es un saldo (una
foto por fecha de corte), así que la función promedia las fotos mensuales en
vez de sumarlas. Sumarlas inflaría la exposición ~12x.

In [ ]:
principales = ["Automóviles", "Vida", "Gastos Médicos (incluye Ind, Gpo y Col)", "Incendio"]

fs = comparar_vs_mercado(filtrar(mercado),
                         aseguradora, frecuencia_severidad)

fs[["DESC_RAMO",
    "frecuencia_pct_inst", "frecuencia_pct_mercado", "frecuencia_pct_dif",
    "severidad_pesos_inst", "severidad_pesos_mercado", "severidad_pesos_dif"]].round(2)

### Crecimiento

Aquí hay que pasarle el `df` **completo**, no `mercado`: para calcular el
crecimiento de 2025 necesita las primas de 2024, y `mercado` ya está filtrado a
un solo año. El recorte al año que nos interesa va al final.

`crecimiento_anual` recorta todos los años a los meses disponibles en el más
reciente. Como la base llega a junio de 2026, las cifras son acumulados
**ene-jun**: comparables entre sí, pero no son año completo. Por eso
`prima_mdp` aquí no coincide con la de las tablas de arriba.

In [ ]:
cre = comparar_vs_mercado(df, aseguradora, crecimiento_anual).query("ANIO == @anio")

cre[["DESC_RAMO", "prima_mdp_inst",
     "crecimiento_pct_inst", "crecimiento_pct_mercado", "crecimiento_pct_dif"]] \
   .sort_values("crecimiento_pct_dif", ascending=False, ignore_index=True) \
   .round(2)

---
## Tabla de benchmarking

Las tres métricas juntas, un renglón por ramo, ordenadas por tamaño de la
cartera. Esta es la tabla que resume la posición competitiva:

- `part_pct` — qué tajada del ramo tiene
- `sin_dif` — resultado técnico contra el mercado (positivo = peor)
- `crec_dif` — si gana o pierde terreno (positivo = crece más rápido)

In [ ]:
perfil_ramo = filtrar(participacion_mercado(mercado), institucion=aseguradora)

resumen = (
    perfil_ramo[["DESC_RAMO", "prima_mdp", "participacion_pct"]]
    .merge(sin[["DESC_RAMO", "siniestralidad_pct_inst",
                "siniestralidad_pct_mercado", "siniestralidad_pct_dif"]], on="DESC_RAMO")
    .merge(cre[["DESC_RAMO", "crecimiento_pct_inst",
                "crecimiento_pct_mercado", "crecimiento_pct_dif"]], on="DESC_RAMO")
    .sort_values("prima_mdp", ascending=False, ignore_index=True)
)

# nombres cortos, solo para que la tabla quepa en pantalla
resumen.columns = ["ramo", "prima_mdp", "part_pct", "sin_sura", "sin_mkt",
                   "sin_dif", "crec_sura", "crec_mkt", "crec_dif"]
resumen.round(2)

### Lectura

- **Automóviles** es su cartera más grande (2,201 mdp) pero solo 1.0% del ramo,
  con siniestralidad en línea con el mercado y una caída de 19% mientras el
  ramo crecía 13%: pierde terreno en su negocio principal.
- **Vida** es su segunda cartera y su peor posición relativa: 0.27% del ramo y
  una siniestralidad 32 puntos arriba del mercado.
- **Gastos Médicos** es la nota positiva: creció 39% contra 8% del mercado y su
  siniestralidad está 4 puntos por debajo.
- **Cascos Embarcaciones** tiene el mercado con siniestralidad **negativa**
  (−75%). No es un error de cálculo: Mapfre reporta un `MONTO_SIN` acumulado de
  −5,953 mdp en Campeche, que ya viene así en la fuente. Cualquier comparación
  contra ese mercado no significa gran cosa.

---
## Visualización: tamaño de cartera contra peso en el ramo

La tentación aquí es una gráfica de barras con las primas y una línea con la
participación encima, sobre dos ejes verticales. **No lo hagas.** La alineación
entre las dos escalas es arbitraria, así que el gráfico inventa una correlación
que no está en los datos: mueves el rango de un eje y la "relación" cambia.

La forma correcta para dos medidas es una **dispersión**: cada medida tiene su
propio eje de posición y la relación se lee sola. El eje X va en escala
logarítmica porque las primas de Sura van de 38 a 2,201 mdp — casi dos órdenes
de magnitud— y en escala lineal los ramos chicos se apelmazan contra el cero.

In [ ]:
from graficas import dispersion_cartera

ax = dispersion_cartera(perfil_ramo, f"{aseguradora} {anio} — tamaño de cartera frente a peso en el ramo")

### Cómo leerla

Las punteadas son las medianas y parten la gráfica en cuadrantes:

- **Abajo a la derecha** — mucha prima, poco peso relativo. Es el cuadrante
  incómodo: Vida y Gastos Médicos son la segunda y tercera cartera de Sura,
  pero ahí es un jugador marginal (0.27% y 0.57%). Automóviles cae en el mismo
  lado: su cartera más grande con apenas 1.0% del ramo.
- **Arriba a la izquierda** — donde más pesa, pero en mercados chicos.
  Accidentes Personales es su mejor participación (2.98%) con solo 206 mdp.
- La gráfica dice que Sura **no tiene un ramo donde sea grande y fuerte a la
  vez**: no hay puntos arriba a la derecha.

La función sirve para cualquier aseguradora. Vale la pena correrla con un
monoproducto para ver el contraste — Quálitas solo opera en 2 ramos, con 33.4%
de Automóviles:

In [ ]:
ax = dispersion_cartera(
    filtrar(part, institucion="Quálitas"),
    "Quálitas 2025 — una aseguradora monoproducto",
)

---
## Correlación entre dos métricas

`dispersion()` grafica dos columnas cualesquiera y calcula su correlación de
rangos (Spearman). Los puntos pueden ser aseguradoras o ramos, según lo que
hayas agregado antes: la función no lo sabe ni lo necesita.

Se usa Spearman y no Pearson porque las primas están muy sesgadas —unas pocas
aseguradoras enormes y una cola larga—, y ahí Pearson termina midiendo sobre
todo el peso de los extremos.

Primera pregunta: **¿ser grande mejora el resultado técnico?** Si hubiera
economías de escala, las aseguradoras con más participación tendrían menor
siniestralidad, y la nube bajaría hacia la derecha.

In [ ]:
from graficas import dispersion

autos = filtrar(df, anio=anio, ramo="Automóviles")

comp = (participacion_mercado(autos)[["NOMBRE_CORTO", "prima_mdp", "participacion_pct"]]
        .merge(siniestralidad(autos)[["NOMBRE_CORTO", "siniestralidad_pct"]], on="NOMBRE_CORTO")
        .merge(frecuencia_severidad(autos)[["NOMBRE_CORTO", "frecuencia_pct", "severidad_pesos"]],
               on="NOMBRE_CORTO"))

# Tres aseguradoras marginales tienen siniestralidad de 200% a 370% y aplastan
# la escala. Se acotan para poder leer la nube; siguen en `comp`.
ax = dispersion(comp[comp.siniestralidad_pct.between(0, 120)],
                "participacion_pct", "siniestralidad_pct",
                etiqueta="NOMBRE_CORTO", log_x=True,
                titulo="Automóviles 2025 — ¿la escala mejora el resultado técnico?",
                nota="Cada punto es una aseguradora. Si el tamaño ayudara, la nube bajaría hacia la derecha.")

**rho = 0.27**, es decir: no baja, sube un poco. En Automóviles el tamaño no
compra un mejor índice de siniestralidad. Las siniestralidades más bajas son de
aseguradoras diminutas, que probablemente apenas operan el ramo.

Con `n = 31` y rho de 0.27, esto es una observación, no una conclusión: el
coeficiente es débil y la muestra chica.

### La trampa: métricas que comparten un término

Correlacionar frecuencia contra severidad parece la comparación natural, pero
está mal por construcción:

    frecuencia = siniestros / expuestos      ← siniestros arriba
    severidad  = monto / siniestros          ← siniestros abajo

Cualquier error al contar siniestros mueve las dos en sentidos opuestos, así
que sale correlación negativa aunque las dos cosas sean independientes. La
función avisa cuando detecta un par así:

In [ ]:
# Lanza un UserWarning explicando por qué esta correlación no significa nada.
ax = dispersion(comp, "frecuencia_pct", "severidad_pesos",
                etiqueta="NOMBRE_CORTO",
                titulo="Ejemplo de correlación espuria — no la interpretes")

La lista de pares vigilados está en `graficas.PARES_ESPURIOS` y no puede ser
exhaustiva: antes de leer cualquier rho, revisa si las dos métricas comparten
alguna columna de la base.

Una comparación que sí es legítima, porque los denominadores son distintos:

In [ ]:
ax = dispersion(comp[comp.siniestralidad_pct.between(0, 120)],
                "frecuencia_pct", "siniestralidad_pct",
                etiqueta="NOMBRE_CORTO",
                titulo="Automóviles 2025 — frecuencia contra siniestralidad",
                nota="Frecuencia usa riesgos expuestos; siniestralidad usa prima. No comparten término.")

---
## ¿Desde qué diferencia hay que preocuparse?

No hay un número fijo que sirva, y conviene ver por qué antes de elegir uno.
Medido sobre los 440 pares institución × ramo de 2025, el rango intercuartil de
la diferencia va de **0.8 puntos en Terremoto a 71 en Incendio**. Estar 20
puntos arriba del mercado es catastrófico en el primero y ordinario en el
segundo. Además, el ruido depende del tamaño: la diferencia mediana absoluta es
de 64 puntos en carteras de menos de 1 mdp y de 10.5 en las de más de 1,000.

`alerta_siniestralidad` mide la distancia en **rangos intercuartiles del propio
ramo** (1.5 = atípico según la convención de Tukey, 3.0 = extremo) y no marca
carteras por debajo de 50 mdp, donde el cociente es ruido.

In [ ]:
from metricas import alerta_siniestralidad

alerta = alerta_siniestralidad(mercado, aseguradora)
alerta.round(1)

### Por qué los puntos porcentuales engañan

Compara las dos primeras filas. Ordenada por diferencia cruda, la peor parece
**Cascos Embarcaciones (+77 puntos)** — pero está a −0.2 IQR: es normal para su
ramo, y además el mercado tiene siniestralidad negativa, así que la comparación
no significa nada. La segunda, **Diversos Misceláneos (+56 puntos)**, sí es un
atípico real: su ramo es apretado (IQR de 17), así que +56 son 3.3 IQR.

Y al revés: **Vida** aparece +32.5 puntos arriba del mercado, que asusta, pero
son 0.7 IQR — perfectamente normal en un ramo tan disperso.

De los 13 ramos de Sura, el único que se marca es Diversos Misceláneos.

### Las dos columnas son señales distintas

- `bandera` es **relativa**: qué tan lejos estás de lo que hacen tus
  competidores en ese ramo.
- `sobre_prima` es **absoluta**: True cuando pagaste más en siniestros que lo
  que emitiste en prima. Es un hecho financiero, no dependa del mercado.

No se colapsan en una sola escala porque son ciertas por separado: FM Global
tuvo 174.9% en Incendio 2025 —pérdida técnica clara— pero a solo 1.9 IQR de sus
competidores, porque Incendio es un ramo muy disperso.

Calibración sobre los 328 pares con volumen de 2025: 3.4% salen "atención",
4.6% "grave" y 7.6% marcan `sobre_prima`.

In [ ]:
# Si prefieres un umbral fijo en puntos para un reporte, estos son los
# percentiles reales de la diferencia en carteras de 50 mdp o más (2025):
#     +13 pts = cuartil superior   (común, no alarmante)
#     +45 pts = decil superior     (atención)
#     +80 pts = percentil 95       (grave)
# Sirven como referencia, pero ignoran que cada ramo tiene su propia dispersión.

alerta.query("bandera in ['grave', 'atención'] or sobre_prima")[
    ["DESC_RAMO", "prima_mdp", "siniestralidad_pct", "mercado_pct", "dif", "iqrs", "bandera"]
].round(1)

### La misma alerta, en gráfica

`alerta_ramos` dibuja el eje en **rangos intercuartiles**, no en puntos
porcentuales, porque es la unidad en que está definido el umbral: así las dos
punteadas *son* los cortes de 1.5 y 3.0, y se ve de inmediato qué tan lejos
quedó cada ramo. Con puntos porcentuales en el eje harían falta trece líneas
distintas, una por ramo.

La diferencia cruda va al margen derecho, para tener las dos lecturas juntas —
que es justo donde se contradicen.

El color de estado nunca va solo: los ramos marcados llevan la palabra al lado,
porque sobre fondo claro estos tonos no alcanzan 3:1 de contraste y el color
por sí mismo no es legible para todos.

In [ ]:
from graficas import alerta_ramos

ax = alerta_ramos(alerta, f"{aseguradora} {anio} — ramos con siniestralidad fuera de lo normal")

Sura tiene un solo ramo marcado. Para ver los cinco estados de la bandera a la
vez conviene una aseguradora más diversificada:

In [ ]:
ax = alerta_ramos(
    alerta_siniestralidad(mercado, "Mapfre México"),
    "Mapfre México 2025 — ramos con siniestralidad fuera de lo normal",
)

Esa gráfica es el mejor argumento para el umbral relativo. Mira **Terremoto**:
+1 punto de diferencia contra el mercado, y sale **grave**. No es un error — el
rango intercuartil de Terremoto es de 0.8 puntos, así que +1 punto son 6.1 IQR.
En ese ramo todas las aseguradoras están pegadas al mismo número, y separarse
aunque sea un punto ya es anómalo.

Y al revés: **Cascos Aeronaves** con +210 puntos apenas llega a "atención",
porque ese ramo es tan volátil que 210 puntos entran dentro de lo normal.

Un umbral fijo en puntos porcentuales habría marcado exactamente al revés.

---
## ¿El problema es frecuencia, severidad, las dos o ninguna?

`diagnostico_frecuencia_severidad` compara los dos factores contra el mercado
como razones —1x es "igual al mercado"— y clasifica cada ramo. `tolerancia`
define qué tan lejos de 1x hay que estar para contar: sin ella, un ramo 3%
arriba saldría diagnosticado.

In [ ]:
from metricas import diagnostico_frecuencia_severidad
from graficas import mapa_frecuencia_severidad

diag = diagnostico_frecuencia_severidad(mercado, aseguradora)
diag.round(2)

En la gráfica **el cuadrante es el diagnóstico**: no hace falta leer una
leyenda de color para saber qué le pasa a un ramo, basta ver en qué esquina
cayó. Los ejes van en escala logarítmica porque son razones — estar al doble y
a la mitad deben verse a la misma distancia del centro.

La diagonal marca el costo técnico igual al del mercado
(frecuencia × severidad = 1). Por encima de ella el riesgo cuesta más que el
promedio del ramo, venga de donde venga.

In [ ]:
ax = mapa_frecuencia_severidad(diag, f"{aseguradora} {anio} — ¿el costo viene de frecuencia o de severidad?")

### Lectura de Sura

- **Div. Misceláneos** está arriba a la derecha y en rojo: frecuencia 11.7x y
  severidad 1.5x. Es el mismo ramo que `alerta_siniestralidad` marcó "grave".
- **Accidentes Personales** y **Agrícola** son problemas de frecuencia pura:
  muchos siniestros, de costo normal o menor. La palanca es suscripción.
- **Vida** es lo contrario: 0.46x de frecuencia pero 1.33x de severidad, y en
  rojo. Pocos siniestros, caros, y no cubiertos por la prima.
- **Gastos Médicos** también es severidad (5.3x) pero **gris**: la prima lo
  refleja y su siniestralidad queda por debajo del mercado.

### El color responde una segunda pregunta

El cuadrante dice de dónde viene el costo. El color dice si eso importa.

**Automóviles** es el mejor ejemplo: cae en "ambos" —frecuencia 1.20x y
severidad 1.23x— pero sale gris. Su costo técnico por riesgo es 1.47x el del
mercado y su prima por riesgo es 1.44x: la siniestralidad queda en 1.02. El
riesgo es más caro **y está cobrado**.

Por eso este mapa describe la composición del riesgo, no la rentabilidad. Un
ramo puede estar en "ambos" y estar perfectamente bien.

### ¿El problema es el riesgo o la tarifa?

`diagnostico` dice de qué factor del riesgo viene el costo. No dice si el
problema es el riesgo o el precio, y no siempre coinciden.

Las tres razones están ligadas por una identidad exacta:

    razón de siniestralidad = razón de costo ÷ razón de prima

Como se dividen, restarlas no significa nada: 17x de costo no se compara con
3x de precio en crudo. En logaritmos la división se vuelve resta, los dos
aportes quedan en la misma escala y **suman exactamente el desajuste total**.
`causa_dominante` se queda con el que pesa más.

In [ ]:
diag[["DESC_RAMO", "diagnostico", "razon_costo", "razon_prima",
      "razon_siniestralidad", "aporte_costo", "aporte_precio", "causa_dominante"]].round(2)

**Vida es el caso que lo justifica.** Sale etiquetada `"severidad"` porque su
severidad es 1.33x, apenas encima de la tolerancia — pero su costo total es
0.62x, o sea **más barato** que el del mercado. Lo que arruina el resultado es
que cobra 0.23x la prima del mercado por riesgo. La etiqueta apunta al lugar
equivocado; `causa_dominante` dice `"precio"`.

Cuando el desajuste de siniestralidad cae dentro de la tolerancia, la columna
dice `"en línea"`: no hay nada que atribuir.

Esto no es un caso aislado. Cruzando las dos columnas en todo el mercado de
2025, sobre los ramos con volumen y siniestralidad arriba del mercado:

| diagnóstico | domina costo | domina precio |
|---|---|---|
| ambos | 27 | 0 |
| frecuencia | 18 | 9 |
| severidad | 16 | **26** |
| ninguno | 0 | 5 |

Los ramos etiquetados `"severidad"` están dominados por el **precio** más veces
que por el costo. Es justo donde `diagnostico` más engaña.

---
## Crecimiento: ¿gana o pierde terreno?

Lo importante de la tabla de crecimiento no es la tasa, es la **brecha**.
Crecer 4% donde el mercado creció 30% es perder terreno, y perder terreno es
literalmente ceder cuota de mercado.

`carrera_contra_mercado` pone las dos tasas como un dumbbell —la barra entre
los dos puntos *es* la brecha— y traduce esa brecha a puntos de participación
en el margen derecho. La traducción es exacta, no estimada: la cuota de cada
año se calcula con las primas que la propia tabla ya trae.

Se muestran las dos tasas y no solo la diferencia porque no son lo mismo: caer
19% mientras el mercado crece 13% es una situación distinta de crecer 4%
mientras el mercado crece 30%, aunque la brecha se parezca.

In [ ]:
from graficas import carrera_contra_mercado

ax = carrera_contra_mercado(cre, f"{aseguradora} {anio} — ¿gana o pierde terreno? (ene-jun contra ene-jun)")

### Lectura

**Sura cede cuota en 8 de 13 ramos.** Y lo hace justo donde más pesa:

- **Automóviles**, su cartera más grande, cae 19% mientras el ramo crece 13%.
  Cede 0.29 puntos de cuota. Es coherente con lo que vimos en el mapa de
  frecuencia y severidad: cobra 1.44x el precio del mercado, y el precio alto
  se paga en volumen.
- **Transportes** crece +4% — un número que aislado se vería bien — pero su
  mercado creció +30%. Ese es el caso que una barra de diferencias escondería
  y el dumbbell hace evidente.
- **Div. Ramos Técnicos** y **Agrícola** son las mayores cesiones de cuota
  (−0.51 y −0.50 puntos).

Del lado bueno, gana terreno en **Gastos Médicos** (+39% contra +8%),
**F. Hidrometeorológicos** (+43% contra +19%) y **Terremoto** — los mismos
ramos que en el mapa anterior salían grises, con el costo bien cobrado.

Cascos Embarcaciones aparece recortado (+99% contra +90%) sobre 3 mdp de
prima: porcentajes enormes sobre una base diminuta, que es exactamente lo que
la compuerta de volumen está para no dejar pasar.

---
# Resumen

`resumen()` recorre todas las métricas del módulo y se queda con el top `n` de
cada una, en los dos sentidos cuando tiene sentido. Devuelve un DataFrame
largo —una fila por titular— con `tema` y `signo` para filtrar, y una columna
`detalle` ya redactada para pegar en un reporte.

Dos cosas que hace por debajo y conviene saber:

- **Escala los umbrales de volumen a la ventana del año elegido.** Si pides
  2026, que en la base llega a junio, el corte de 50 mdp pasa a 25 y la
  función lanza un aviso de que las cifras son de medio año.
- **No inventa fortalezas.** Un titular de siniestralidad favorable exige
  estar al menos medio IQR por debajo de la mediana del ramo. Si la
  aseguradora no tiene ninguno, esa sección sale vacía — que es en sí mismo
  un hallazgo.

In [ ]:
from metricas import resumen

titulares = resumen(df, aseguradora, anio=anio, n=3)
titulares

Y la misma tabla como texto, agrupada por tema:

In [ ]:
ancho = 78
print(f"{aseguradora} — {anio}".center(ancho))
print("=" * ancho)

for (tema, signo), grupo in titulares.groupby(["tema", "signo"], sort=False):
    print(f"\n{tema.upper()}  ({signo})")
    for linea in grupo.detalle:
        print(f"   · {linea}")

vacios = {"siniestralidad"} - set(titulares.query("signo == 'favorable'").tema)
if vacios:
    print(f"\nSin titulares favorables de: {', '.join(vacios)}")

La gracia de que devuelva un DataFrame y no texto es que se puede recortar:

```python
titulares.query("signo == 'desfavorable'")        # solo lo que hay que arreglar
titulares.query("tema == 'crecimiento'")          # solo la parte comercial
resumen(df, "Quálitas", anio=anio, n=5)           # otra aseguradora, top 5
```

In [ ]:
titulares.query("signo == 'desfavorable'")[["tema", "DESC_RAMO", "detalle"]]

---
## Concentración de cartera

Contar ramos dice en cuántos opera una aseguradora, no cuánto pesa cada uno.
Sura y AXA operan las dos en **13 ramos** y el conteo las iguala, pero no están
igual de diversificadas.

`concentracion_cartera` lo mide con el índice de Herfindahl-Hirschman sobre la
cartera propia — la suma de los cuadrados de las cuotas internas:

$$HHI = \sum_i s_i^2 \qquad s_i = \frac{\text{prima del ramo } i}{\text{prima total}}$$

Va de $1/n$ (todo repartido parejo entre $n$ ramos) a $1$ (todo en uno solo).
Su inverso, $1/HHI$, son los **ramos efectivos**: cuántos ramos de igual tamaño
darían esa misma concentración. Es la cifra que se lee sin explicar nada.

In [ ]:
from metricas import concentracion_cartera

conc = concentracion_cartera(mercado)
conc[conc.prima_mdp > 20000].round(2)

Ahí está la diferencia que el conteo escondía: **Sura reparte su prima como si
tuviera 6 ramos y AXA como si tuviera 3.4**, con los mismos 13 cada una. Y
Quálitas sale en 1.00 aunque opere en dos, porque su segundo ramo son 2 mdp de
73,474 — el conteo dice 2, la realidad es monoproducto.

Importa porque condiciona cómo se leen las demás métricas: la siniestralidad
global de Quálitas *es* su siniestralidad de Automóviles, mientras que la de
Sura es una mezcla de trece. Compararlas sin saber esto induce a error.

El parámetro `por` mide otra dimensión. Con entidad federativa sale la
concentración geográfica —válida para primas, a diferencia de la siniestralidad
por estado, que está rota porque la póliza se emite en un estado y el siniestro
se registra en otro:

In [ ]:
geo = concentracion_cartera(mercado, por=("DESC_ENTIDADFEDERATIVA",))
geo[geo.NOMBRE_CORTO.isin([aseguradora, "AXA Seguros", "Quálitas"])].round(2)

Se invierte el orden: **Quálitas es la más concentrada por ramo y la más
repartida por geografía** (15.5 estados efectivos contra 6.2 de Sura). Vender un
solo producto en todo el país es un perfil de riesgo distinto al de vender
muchos productos concentrados en pocos estados.